<a href="https://colab.research.google.com/github/yhou151209/541_project/blob/main/541.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install ortools

In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Small demo dataset for Phase 1.
    You can replace this with JSON-loaded data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "max_shifts": 2,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "max_shifts": 3,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "max_shifts": 2,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "max_shifts": 2,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "max_shifts": 3,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            # Monday
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},

            # Tuesday
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        key = (row["employee_id"], row["day"], row["shift"])
        lookup[key] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Build a schedule from scratch with OR-Tools CP-SAT.

    Hard constraints:
    1. Required coverage must be met exactly.
    2. Employee must be available for assigned shift.
    3. Employee must have the required skill/role.
    4. Employee cannot hold two roles in the same day/shift.
    5. Employee cannot exceed max_shifts.

    Soft objective:
    - Minimize the maximum number of assigned shifts to keep load more balanced.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)

    model = cp_model.CpModel()

    # Decision variable:
    # x[(emp_id, day, shift, role)] = 1 if employee is assigned to that role in that shift.
    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            emp_skills = set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)
            has_skill = role in emp_skills

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            # Not available or not qualified -> cannot assign
            if (not is_available) or (not has_skill):
                model.Add(var == 0)

    # Coverage constraints: exactly meet required_count
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        eligible_vars = [
            x[(emp["id"], day, shift, role)]
            for emp in staff
        ]
        model.Add(sum(eligible_vars) == required_count)

    # Same employee cannot take two roles in the same shift
    unique_shift_keys = sorted({(r["day"], r["shift"]) for r in requirements})
    roles = sorted({r["role"] for r in requirements})

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in unique_shift_keys:
            vars_same_shift = [
                x[(emp_id, day, shift, role)]
                for role in roles
                if (emp_id, day, shift, role) in x
            ]
            if vars_same_shift:
                model.Add(sum(vars_same_shift) <= 1)

    # Max shifts per employee
    for emp in staff:
        emp_id = emp["id"]
        max_shifts = int(emp["max_shifts"])
        emp_vars = [
            var for key, var in x.items()
            if key[0] == emp_id
        ]
        model.Add(sum(emp_vars) <= max_shifts)

    # Soft fairness objective:
    # minimize the maximum assigned shifts among employees
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    for emp in staff:
        emp_id = emp["id"]
        total = model.NewIntVar(0, len(unique_shift_keys), f"total_{emp_id}")
        emp_vars = [
            var for key, var in x.items()
            if key[0] == emp_id
        ]
        model.Add(total == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total

    max_load = model.NewIntVar(0, len(unique_shift_keys), "max_load")
    for emp_id, total in total_assigned_per_emp.items():
        model.Add(total <= max_load)

    model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found with current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def print_schedule(schedule: List[Dict[str, str]]) -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print("\n===== SCHEDULE =====")
    for day_shift in sorted(grouped.keys()):
        day, shift = day_shift
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[day_shift], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("====================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> List[Dict[str, str]]:
    """
    Simple Phase 1 update:
    supports requests like:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Strategy:
    1. Mark that employee unavailable for the target shift.
    2. Re-run schedule generation from scratch.
       This is the simplest robust Phase 1 approach.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"]
    day = change["day"]
    shift = change["shift"]

    # Find employee id
    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    # Copy data deeply enough for this use case
    new_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found_availability_row = False
    for row in new_data["availability"]:
        if row["employee_id"] == emp_id and row["day"] == day and row["shift"] == shift:
            row["available"] = False
            found_availability_row = True

    if not found_availability_row:
        new_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    # Regenerate whole schedule for simplicity
    updated_schedule = generate_schedule(new_data)
    return updated_schedule


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule)

    # Manual Phase 1 test: Amy unavailable Monday evening
    change_request = {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening",
    }

    print("Applying update: Amy unavailable Monday evening...")
    updated = update_schedule(data, schedule, change_request)
    print_schedule(updated)


if __name__ == "__main__":
    main()

Generating initial schedule...

===== SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> John (E02)

Monday - morning
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> David (E05)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> David (E05)

Applying update: Amy unavailable Monday evening...

===== SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> Betty (E03)
  server   -> Amy (E01)

Tuesday - evening
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> David (E05)
  server   -> Cathy (E04)



In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Demo restaurant dataset for Phase 1.
    Replace this with your real restaurant data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E06",
                "name": "Emma",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 2,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": False},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E06", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E06", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(schedule: List[Dict[str, str]]) -> set[Tuple[str, str, str, str]]:
    """
    Convert schedule list into a set of:
    (employee_id, day, shift, role)
    """
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    General solver.

    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    # Exact coverage
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    # One employee cannot do two roles in same shift
    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    # Max shifts
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    # Fairness: minimize max workload
    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for emp_id, total_var in total_assigned_per_emp.items():
        model.Add(total_var <= max_load)

    # Keep-as-many-existing-assignments-as-possible objective
    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    # Lexicographic-style weighted objective
    # Strongly prefer keeping old assignments, secondarily reduce max load
    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    """
    Currently supports:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Update strategy:
    1. Modify availability.
    2. Re-solve, but use existing_schedule as a preference,
       so the solver tries to keep as many old assignments as possible.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"].strip()
    day = change["day"].strip()
    shift = change["shift"].strip()

    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found = False
    for row in updated_data["availability"]:
        if row["employee_id"] == emp_id and row["day"].lower() == day.lower() and row["shift"].lower() == shift.lower():
            row["available"] = False
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        updated_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
    return updated_schedule, updated_data


def prompt_unavailable_change() -> Dict[str, str]:
    """
    Command-line input for Phase 1 testing.
    """
    print("Enter an update request.")
    employee_name = input("Employee name: ").strip()
    day = input("Day (e.g. Monday): ").strip()
    shift = input("Shift (e.g. morning/evening): ").strip()

    return {
        "type": "unavailable",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
    }


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule, title="INITIAL SCHEDULE")

    answer = input("Do you want to enter an unavailable update? (y/n): ").strip().lower()
    if answer != "y":
        print("Done.")
        return

    try:
        change_request = prompt_unavailable_change()
        updated_schedule, updated_data = update_schedule(data, schedule, change_request)

        print_schedule(updated_schedule, title="UPDATED SCHEDULE")
        compare_schedules(schedule, updated_schedule)

        # Optional: update in-memory state so more edits can be chained later
        data = updated_data
        schedule = updated_schedule

    except ValueError as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    main()

Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

Do you want to enter an unavailable update? (y/n): y
Enter an update request.
Employee name: Emma
Day (e.g. Monday): Tuesday
Shift (e.g. morning/evening): morning

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> David (E05)
  server   -> Amy (E01)

===== CHANGES =====
Removed assignments:
  - E06: Tuesday morning cashier
Added assignments:
  + E05: Tuesday morning cashier



In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def build_sample_data() -> Dict[str, Any]:
    """
    Demo restaurant dataset for Phase 1.
    Replace this with your real restaurant data later.
    """
    return {
        "staff": [
            {
                "id": "E01",
                "name": "Amy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E02",
                "name": "John",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E03",
                "name": "Betty",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E04",
                "name": "Cathy",
                "skills": ["server"],
                "employment_type": "part-time",
                "max_shifts": 3,
            },
            {
                "id": "E05",
                "name": "David",
                "skills": ["server", "cashier"],
                "employment_type": "full-time",
                "max_shifts": 4,
            },
            {
                "id": "E06",
                "name": "Emma",
                "skills": ["cashier"],
                "employment_type": "part-time",
                "max_shifts": 2,
            },
        ],
        "availability": [
            {"employee_id": "E01", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E01", "day": "Tuesday", "shift": "evening", "available": False},

            {"employee_id": "E02", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E02", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E03", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E03", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E03", "day": "Tuesday", "shift": "morning", "available": False},
            {"employee_id": "E03", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E04", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E04", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E04", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E05", "day": "Monday", "shift": "morning", "available": False},
            {"employee_id": "E05", "day": "Monday", "shift": "evening", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E05", "day": "Tuesday", "shift": "evening", "available": True},

            {"employee_id": "E06", "day": "Monday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Monday", "shift": "evening", "available": False},
            {"employee_id": "E06", "day": "Tuesday", "shift": "morning", "available": True},
            {"employee_id": "E06", "day": "Tuesday", "shift": "evening", "available": True},
        ],
        "shift_requirements": [
            {"day": "Monday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Monday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Monday", "shift": "evening", "role": "server", "required_count": 2},
            {"day": "Monday", "shift": "evening", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "morning", "role": "cashier", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "server", "required_count": 1},
            {"day": "Tuesday", "shift": "evening", "role": "cashier", "required_count": 1},
        ],
    }


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(schedule: List[Dict[str, str]]) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    General solver.

    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for emp_id, total_var in total_assigned_per_emp.items():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    """
    Currently supports:
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    Update strategy:
    1. Modify availability.
    2. Re-solve, but use existing_schedule as a preference,
       so the solver tries to keep as many old assignments as possible.
    """
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"].strip()
    day = change["day"].strip()
    shift = change["shift"].strip()

    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found = False
    for row in updated_data["availability"]:
        if (
            row["employee_id"] == emp_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = False
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        updated_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
    return updated_schedule, updated_data


def prompt_unavailable_change() -> Dict[str, str]:
    print("\nEnter an unavailable update request.")
    employee_name = input("Employee name: ").strip()
    day = input("Day (e.g. Monday): ").strip()
    shift = input("Shift (e.g. morning/evening): ").strip()

    return {
        "type": "unavailable",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data = build_sample_data()

    print("Generating initial schedule...")
    schedule = generate_schedule(data)
    print_schedule(schedule, title="INITIAL SCHEDULE")

    while True:
        should_update = ask_yes_no("Do you want to enter an unavailable update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_unavailable_change()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            # Keep latest state for the next round
            data = updated_data
            schedule = updated_schedule

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")


if __name__ == "__main__":
    main()

Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

Do you want to enter an unavailable update? (y/n): y

Enter an unavailable update request.
Employee name: Betty
Day (e.g. Monday): Tuesday
Shift (e.g. morning/evening): evening

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Tuesday - evening
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

===== CHANGES =====
Removed assignments:
  - E03: Tuesday evening cashier
Added assignments:
  + E06: Tuesday evening cashier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")


def make_availability_lookup(availability: List[Dict[str, Any]]) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(schedule: List[Dict[str, str]]) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    if change.get("type") != "unavailable":
        raise ValueError("Currently only 'unavailable' updates are supported.")

    employee_name = change["employee_name"].strip()
    day = change["day"].strip()
    shift = change["shift"].strip()

    matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
    if not matched:
        raise ValueError(f"Employee '{employee_name}' not found.")

    emp_id = matched[0]["id"]

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
    }

    found = False
    for row in updated_data["availability"]:
        if (
            row["employee_id"] == emp_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = False
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        updated_data["availability"].append(
            {
                "employee_id": emp_id,
                "day": day,
                "shift": shift,
                "available": False,
            }
        )

    updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
    return updated_schedule, updated_data


def prompt_unavailable_change() -> Dict[str, str]:
    print("\nEnter an unavailable update request.")
    employee_name = input("Employee name: ").strip()
    day = input("Day (e.g. Monday): ").strip()
    shift = input("Shift (e.g. morning/evening): ").strip()

    return {
        "type": "unavailable",
        "employee_name": employee_name,
        "day": day,
        "shift": shift,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")
    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")

    while True:
        should_update = ask_yes_no("Do you want to enter an unavailable update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_unavailable_change()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updated availability back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> David (E05)

Monday - morning
  cashier  -> Emma (E06)
  server   -> John (E02)

Tuesday - evening
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> Cathy (E04)

Do you want to enter an unavailable update? (y/n): y

Enter an unavailable update request.
Employee name: Cathy
Day (e.g. Monday): Tuesday
Shift (e.g. morning/evening): morning

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> David (E05)

Monday - morning
  cashier  -> Emma (E06)
  server   -> John (E02)

Tuesday - evening
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> John (E02)
  server   -> Amy (E01)

===== CHANGES =====
Removed assignments:
  

In [23]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def is_employee_assigned(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> bool:
    return len(get_assignments_for_employee_shift(schedule, employee_name, day, shift)) > 0


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.
    6. Forced assignments must be included.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]

    staff_lookup = get_staff_lookup(staff)
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    # Exact coverage
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    # One employee cannot do two roles in same shift
    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    # Max shifts
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    # Forced assignments
    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    # Fairness
    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    # Prefer keeping old assignments
    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 100 - max_load)
    else:
        model.Minimize(max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, str],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    """
    Supported change types:

    1. unavailable
    {
        "type": "unavailable",
        "employee_name": "Amy",
        "day": "Monday",
        "shift": "evening"
    }

    2. direct_swap
    {
        "type": "direct_swap",
        "employee_name_1": "Betty",
        "day_1": "Saturday",
        "shift_1": "morning",
        "employee_name_2": "John",
        "day_2": "Tuesday",
        "shift_2": "evening"
    }

    direct_swap logic:
    - employee 1 must currently be assigned to shift 1
    - employee 2 must currently be assigned to shift 2
    - after swap:
        employee 1 must take employee 2's role on shift 2
        employee 2 must take employee 1's role on shift 1
    - both must be available for their new shifts
    - both must have the right skills for the swapped roles
    """
    change_type = change.get("type", "").strip()

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]

        updated_data = {
            "staff": [dict(s) for s in data["staff"]],
            "availability": [dict(a) for a in data["availability"]],
            "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        }

        set_availability(updated_data, emp_id, day, shift, False)
        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = change["day_1"].strip()
        shift_1 = change["shift_1"].strip()

        name_2 = change["employee_name_2"].strip()
        day_2 = change["day_2"].strip()
        shift_2 = change["shift_2"].strip()

        matched_1 = [s for s in data["staff"] if s["name"].lower() == name_1.lower()]
        matched_2 = [s for s in data["staff"] if s["name"].lower() == name_2.lower()]

        if not matched_1:
            raise ValueError(f"Employee '{name_1}' not found.")
        if not matched_2:
            raise ValueError(f"Employee '{name_2}' not found.")

        emp_1 = matched_1[0]
        emp_2 = matched_2[0]

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")

        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        updated_data = {
            "staff": [dict(s) for s in data["staff"]],
            "availability": [dict(a) for a in data["availability"]],
            "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        }

        # Make sure each employee can work the other person's shift
        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable' and 'direct_swap'."
    )


def prompt_change_request() -> Dict[str, str]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. direct_swap")

    while True:
        choice = input("Enter 1 or 2: ").strip()
        if choice in {"1", "2"}:
            break
        print("Please enter 1 or 2.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    employee_name_1 = input("Employee 1 name: ").strip()
    day_1 = input("Employee 1 current day: ").strip()
    shift_1 = input("Employee 1 current shift: ").strip()

    employee_name_2 = input("Employee 2 name: ").strip()
    day_2 = input("Employee 2 current day: ").strip()
    shift_2 = input("Employee 2 current shift: ").strip()

    return {
        "type": "direct_swap",
        "employee_name_1": employee_name_1,
        "day_1": day_1,
        "shift_1": shift_1,
        "employee_name_2": employee_name_2,
        "day_2": day_2,
        "shift_2": shift_2,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")

    while True:
        should_update = ask_yes_no("Do you want to enter an update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_change_request()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updated availability back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> Amy (E01)
  server   -> David (E05)

Monday - morning
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Saturday - morning
  cashier  -> Betty (E03)
  server   -> John (E02)

Tuesday - evening
  cashier  -> David (E05)
  server   -> John (E02)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)

Do you want to enter an update? (y/n): y

Choose an update type:
1. unavailable
2. direct_swap
Enter 1 or 2: 2
Employee 1 name: Emma
Employee 1 current day: Tuesday
Employee 1 current shift: cashier
Employee 2 name: Betty
Employee 2 current day: Saturday
Employee 2 current shift: morning
Error: Emma is not currently assigned to Tuesday cashier.
Do you want to print the current final schedule again? (y/n): y

===== CURRENT FINAL SCHEDULE =====

Monday - evening
  cashier  -> Be

In [24]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def is_employee_assigned(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> bool:
    return len(get_assignments_for_employee_shift(schedule, employee_name, day, shift)) > 0


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    if "preferences" not in data:
        data["preferences"] = []

    data["preferences"].append(preference)


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.
    6. Forced assignments must be included.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    C. Minimize preference penalties:
       - avoid_back_to_back: avoid morning + evening on same day
       - avoid_shift: avoid assigning a specific shift type (e.g. evening)
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {s["name"].lower(): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = sorted({r["day"] for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    # Exact coverage
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    # One employee cannot do two roles in the same shift
    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    # Max shifts
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    # Forced assignments
    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    # Fairness
    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    # Prefer keeping old assignments
    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    # Preference penalties
    penalty_terms: List[cp_model.LinearExpr] = []

    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = pref["employee_name"].lower()

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]
        penalty = int(pref.get("penalty", 1))

        if pref_type == "avoid_shift":
            target_shift = pref.get("shift")
            if not target_shift:
                continue

            for role in all_roles:
                for day in all_days:
                    key = (emp_id, day, target_shift, role)
                    if key in x:
                        penalty_terms.append(x[key] * penalty)

        elif pref_type == "avoid_back_to_back":
            # Here back-to-back means morning + evening on the same day
            for day in all_days:
                morning_assigned = model.NewBoolVar(f"morning_assigned_{emp_id}_{day}_{i}")
                evening_assigned = model.NewBoolVar(f"evening_assigned_{emp_id}_{day}_{i}")
                back_to_back = model.NewBoolVar(f"back_to_back_{emp_id}_{day}_{i}")

                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if morning_vars:
                    model.Add(sum(morning_vars) >= 1).OnlyEnforceIf(morning_assigned)
                    model.Add(sum(morning_vars) == 0).OnlyEnforceIf(morning_assigned.Not())
                else:
                    model.Add(morning_assigned == 0)

                if evening_vars:
                    model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                    model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())
                else:
                    model.Add(evening_assigned == 0)

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(back_to_back)
                model.AddBoolOr([morning_assigned.Not(), evening_assigned.Not()]).OnlyEnforceIf(back_to_back.Not())

                penalty_terms.append(back_to_back * penalty)

    total_penalty = model.NewIntVar(0, 10000, "total_penalty")
    if penalty_terms:
        model.Add(total_penalty == sum(penalty_terms))
    else:
        model.Add(total_penalty == 0)

    # Objective priority:
    # 1. keep old assignments
    # 2. reduce preference penalty
    # 3. reduce max workload
    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(grouped.keys()):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def print_preferences(data: Dict[str, Any]) -> None:
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    change_type = change.get("type", "").strip()

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability(updated_data, emp_id, day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = change["day_1"].strip()
        shift_1 = change["shift_1"].strip()

        name_2 = change["employee_name_2"].strip()
        day_2 = change["day_2"].strip()
        shift_2 = change["shift_2"].strip()

        matched_1 = [s for s in data["staff"] if s["name"].lower() == name_1.lower()]
        matched_2 = [s for s in data["staff"] if s["name"].lower() == name_2.lower()]

        if not matched_1:
            raise ValueError(f"Employee '{name_1}' not found.")
        if not matched_2:
            raise ValueError(f"Employee '{name_2}' not found.")

        emp_1 = matched_1[0]
        emp_2 = matched_2[0]

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")

        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    if change_type == "avoid_back_to_back":
        employee_name = change["employee_name"].strip()
        penalty = int(change.get("penalty", 5))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_back_to_back",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "avoid_shift":
        employee_name = change["employee_name"].strip()
        target_shift = change["shift"].strip()
        penalty = int(change.get("penalty", 3))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_shift",
                "employee_name": employee_name,
                "shift": target_shift,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable', 'direct_swap', 'avoid_back_to_back', and 'avoid_shift'."
    )


def prompt_change_request() -> Dict[str, Any]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. direct_swap")
    print("3. avoid_back_to_back")
    print("4. avoid_shift")

    while True:
        choice = input("Enter 1, 2, 3, or 4: ").strip()
        if choice in {"1", "2", "3", "4"}:
            break
        print("Please enter 1, 2, 3, or 4.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name_1 = input("Employee 1 name: ").strip()
        day_1 = input("Employee 1 current day: ").strip()
        shift_1 = input("Employee 1 current shift: ").strip()

        employee_name_2 = input("Employee 2 name: ").strip()
        day_2 = input("Employee 2 current day: ").strip()
        shift_2 = input("Employee 2 current shift: ").strip()

        return {
            "type": "direct_swap",
            "employee_name_1": employee_name_1,
            "day_1": day_1,
            "shift_1": shift_1,
            "employee_name_2": employee_name_2,
            "day_2": day_2,
            "shift_2": shift_2,
        }

    if choice == "3":
        employee_name = input("Employee name: ").strip()
        penalty_text = input("Penalty weight (default 5): ").strip()
        penalty = int(penalty_text) if penalty_text else 5

        return {
            "type": "avoid_back_to_back",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    shift = input("Shift to avoid (e.g. evening): ").strip()
    penalty_text = input("Penalty weight (default 3): ").strip()
    penalty = int(penalty_text) if penalty_text else 3

    return {
        "type": "avoid_shift",
        "employee_name": employee_name,
        "shift": shift,
        "penalty": penalty,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")
    print_preferences(data)

    while True:
        should_update = ask_yes_no("Do you want to enter an update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_change_request()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)
            print_preferences(updated_data)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")
            print_preferences(data)


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Saturday - morning
  cashier  -> John (E02)
  server   -> Amy (E01)

Tuesday - evening
  cashier  -> Betty (E03)
  server   -> Cathy (E04)

Tuesday - morning
  cashier  -> Emma (E06)
  server   -> Amy (E01)


===== CURRENT PREFERENCES =====
No preferences.

Do you want to enter an update? (y/n): y

Choose an update type:
1. unavailable
2. direct_swap
3. avoid_back_to_back
4. avoid_shift
Enter 1, 2, 3, or 4: 3
Employee name: Emma
Penalty weight (default 5): 1

===== UPDATED SCHEDULE =====

Monday - evening
  cashier  -> Betty (E03)
  server   -> David (E05)
  server   -> John (E02)

Monday - morning
  cashier  -> Emma (E06)
  server   -> Cathy (E04)

Saturday - morning
  cashier  -

In [37]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts", "employment_type"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    if "preferences" not in data:
        data["preferences"] = []

    data["preferences"].append(preference)


def is_busy_shift(day: str, shift: str) -> bool:
    if shift == "evening":
        return True
    if day in {"Saturday", "Sunday"} and shift in {"morning", "evening"}:
        return True
    return False


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.
    6. Forced assignments must be included.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    C. Minimize preference penalties.
    D. Prefer full-time staff on busy shifts.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {s["name"].lower(): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = sorted({r["day"] for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    penalty_terms: List[cp_model.LinearExpr] = []

    # Existing preferences
    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = pref["employee_name"].lower()

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]
        penalty = int(pref.get("penalty", 1))

        if pref_type == "avoid_shift":
            target_shift = pref.get("shift")
            if not target_shift:
                continue

            for role in all_roles:
                for day in all_days:
                    key = (emp_id, day, target_shift, role)
                    if key in x:
                        penalty_terms.append(x[key] * penalty)

        elif pref_type == "avoid_back_to_back":
            for day in all_days:
                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if not morning_vars or not evening_vars:
                    continue

                morning_assigned = model.NewBoolVar(f"morning_assigned_{emp_id}_{day}_{i}")
                evening_assigned = model.NewBoolVar(f"evening_assigned_{emp_id}_{day}_{i}")
                back_to_back = model.NewBoolVar(f"back_to_back_{emp_id}_{day}_{i}")

                model.Add(sum(morning_vars) >= 1).OnlyEnforceIf(morning_assigned)
                model.Add(sum(morning_vars) == 0).OnlyEnforceIf(morning_assigned.Not())

                model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(back_to_back)
                model.AddBoolOr([morning_assigned.Not(), evening_assigned.Not()]).OnlyEnforceIf(back_to_back.Not())

                penalty_terms.append(back_to_back * penalty)

    # New business objective:
    # prefer full-time staff on busy shifts
    busy_shift_penalty_terms: List[cp_model.LinearExpr] = []
    busy_penalty_weight = 4

    for (emp_id, day, shift, role), var in x.items():
        emp_type = staff_lookup[emp_id]["employment_type"]
        if is_busy_shift(day, shift) and emp_type != "full-time":
            busy_shift_penalty_terms.append(var * busy_penalty_weight)

    total_penalty = model.NewIntVar(0, 50000, "total_penalty")
    all_penalties = penalty_terms + busy_shift_penalty_terms

    if all_penalties:
        model.Add(total_penalty == sum(all_penalties))
    else:
        model.Add(total_penalty == 0)

    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)

def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    day_order = {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7,
    }

    shift_order = {
        "morning": 1,
        "evening": 2,
    }

    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(
        grouped.keys(),
        key=lambda x: (day_order.get(x[0], 999), shift_order.get(x[1], 999))
    ):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()

def print_preferences(data: Dict[str, Any]) -> None:
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    change_type = change.get("type", "").strip()

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability(updated_data, emp_id, day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "avoid_back_to_back":
        employee_name = change["employee_name"].strip()
        penalty = int(change.get("penalty", 5))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_back_to_back",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "avoid_shift":
        employee_name = change["employee_name"].strip()
        target_shift = change["shift"].strip()
        penalty = int(change.get("penalty", 3))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_shift",
                "employee_name": employee_name,
                "shift": target_shift,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable', 'avoid_back_to_back', and 'avoid_shift'."
    )


def prompt_change_request() -> Dict[str, Any]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. avoid_back_to_back")
    print("3. avoid_shift")

    while True:
        choice = input("Enter 1, 2, or 3: ").strip()
        if choice in {"1", "2", "3"}:
            break
        print("Please enter 1, 2, or 3.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name = input("Employee name: ").strip()
        penalty_text = input("Penalty weight (default 5): ").strip()
        penalty = int(penalty_text) if penalty_text else 5

        return {
            "type": "avoid_back_to_back",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    shift = input("Shift to avoid (e.g. evening): ").strip()
    penalty_text = input("Penalty weight (default 3): ").strip()
    penalty = int(penalty_text) if penalty_text else 3

    return {
        "type": "avoid_shift",
        "employee_name": employee_name,
        "shift": shift,
        "penalty": penalty,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")
    print_preferences(data)

    while True:
        should_update = ask_yes_no("Do you want to enter an update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_change_request()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)
            print_preferences(updated_data)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")
            print_preferences(data)


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Monday - evening
  cashier  -> Brian (E02)
  server   -> Eric (E05)
  server   -> George (E07)

Tuesday - morning
  cashier  -> Julia (E10)
  server   -> Daniel (E04)

Tuesday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> George (E07)

Wednesday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Wednesday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Thursday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Thursday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> Eric (E05)

Friday - morning
  cashier  -> Hannah (E08)
  server   -> Alice (E01)

Friday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Saturday - mor

In [38]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple
from ortools.sat.python import cp_model


def load_data_from_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    validate_data(data)
    return data


def save_data_to_json(data: Dict[str, Any], file_path: str) -> None:
    path = Path(file_path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def validate_data(data: Dict[str, Any]) -> None:
    required_top_keys = {"staff", "availability", "shift_requirements"}
    missing = required_top_keys - set(data.keys())
    if missing:
        raise ValueError(f"Missing top-level keys: {missing}")

    for staff_member in data["staff"]:
        for key in ["id", "name", "skills", "max_shifts", "employment_type"]:
            if key not in staff_member:
                raise ValueError(f"Staff record missing '{key}': {staff_member}")

    for row in data["availability"]:
        for key in ["employee_id", "day", "shift", "available"]:
            if key not in row:
                raise ValueError(f"Availability record missing '{key}': {row}")

    for req in data["shift_requirements"]:
        for key in ["day", "shift", "role", "required_count"]:
            if key not in req:
                raise ValueError(f"Shift requirement missing '{key}': {req}")

    if "preferences" in data:
        for pref in data["preferences"]:
            if "type" not in pref or "employee_name" not in pref:
                raise ValueError(f"Preference missing required fields: {pref}")


def make_availability_lookup(
    availability: List[Dict[str, Any]]
) -> Dict[Tuple[str, str, str], bool]:
    lookup: Dict[Tuple[str, str, str], bool] = {}
    for row in availability:
        lookup[(row["employee_id"], row["day"], row["shift"])] = bool(row["available"])
    return lookup


def get_staff_lookup(staff: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {s["id"]: s for s in staff}


def schedule_to_assignment_set(
    schedule: List[Dict[str, str]]
) -> set[Tuple[str, str, str, str]]:
    return {
        (row["employee_id"], row["day"], row["shift"], row["role"])
        for row in schedule
    }


def get_assignments_for_employee_shift(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[Dict[str, str]]:
    matches = []
    for row in schedule:
        if (
            row["employee_name"].lower() == employee_name.lower()
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            matches.append(row)
    return matches


def get_employee_assignment_roles(
    schedule: List[Dict[str, str]],
    employee_name: str,
    day: str,
    shift: str,
) -> List[str]:
    return [
        row["role"]
        for row in get_assignments_for_employee_shift(schedule, employee_name, day, shift)
    ]


def set_availability(
    data: Dict[str, Any],
    employee_id: str,
    day: str,
    shift: str,
    available: bool,
) -> None:
    found = False
    for row in data["availability"]:
        if (
            row["employee_id"] == employee_id
            and row["day"].lower() == day.lower()
            and row["shift"].lower() == shift.lower()
        ):
            row["available"] = available
            row["day"] = day
            row["shift"] = shift
            found = True

    if not found:
        data["availability"].append(
            {
                "employee_id": employee_id,
                "day": day,
                "shift": shift,
                "available": available,
            }
        )


def add_preference(data: Dict[str, Any], preference: Dict[str, Any]) -> None:
    if "preferences" not in data:
        data["preferences"] = []
    data["preferences"].append(preference)


def is_busy_shift(day: str, shift: str) -> bool:
    if shift == "evening":
        return True
    if day in {"Saturday", "Sunday"} and shift in {"morning", "evening"}:
        return True
    return False


def solve_schedule(
    data: Dict[str, Any],
    preferred_schedule: List[Dict[str, str]] | None = None,
    forced_assignments: List[Tuple[str, str, str, str]] | None = None,
) -> List[Dict[str, str]]:
    """
    Hard constraints:
    1. Meet required coverage exactly.
    2. Employee must be available.
    3. Employee must have required skill.
    4. Employee cannot hold two roles in the same shift.
    5. Employee cannot exceed max_shifts.
    6. Forced assignments must be included.

    Soft objectives:
    A. Prefer keeping assignments from preferred_schedule.
    B. Minimize maximum workload for fairness.
    C. Minimize preference penalties.
    D. Prefer full-time staff on busy shifts.
    """
    staff = data["staff"]
    availability = data["availability"]
    requirements = data["shift_requirements"]
    preferences = data.get("preferences", [])

    staff_lookup = get_staff_lookup(staff)
    name_to_id = {s["name"].lower(): s["id"] for s in staff}
    avail_lookup = make_availability_lookup(availability)
    preferred_assignments = schedule_to_assignment_set(preferred_schedule or [])
    forced_assignments = forced_assignments or []

    model = cp_model.CpModel()

    x: Dict[Tuple[str, str, str, str], cp_model.IntVar] = {}

    all_roles = sorted({r["role"] for r in requirements})
    all_day_shift_pairs = sorted({(r["day"], r["shift"]) for r in requirements})
    all_days = sorted({r["day"] for r in requirements})

    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]

        for emp in staff:
            emp_id = emp["id"]
            has_skill = role in set(emp["skills"])
            is_available = avail_lookup.get((emp_id, day, shift), False)

            var = model.NewBoolVar(f"x_{emp_id}_{day}_{shift}_{role}")
            x[(emp_id, day, shift, role)] = var

            if not has_skill or not is_available:
                model.Add(var == 0)

    # Exact coverage
    for req in requirements:
        day = req["day"]
        shift = req["shift"]
        role = req["role"]
        required_count = int(req["required_count"])

        vars_for_req = [x[(emp["id"], day, shift, role)] for emp in staff]
        model.Add(sum(vars_for_req) == required_count)

    # One employee cannot do two roles in the same shift
    for emp in staff:
        emp_id = emp["id"]
        for day, shift in all_day_shift_pairs:
            same_shift_vars = [
                x[(emp_id, day, shift, role)]
                for role in all_roles
                if (emp_id, day, shift, role) in x
            ]
            if same_shift_vars:
                model.Add(sum(same_shift_vars) <= 1)

    # Max shifts and workload tracking
    total_assigned_per_emp: Dict[str, cp_model.IntVar] = {}
    max_possible_assignments = len(all_day_shift_pairs)

    for emp in staff:
        emp_id = emp["id"]
        emp_vars = [var for key, var in x.items() if key[0] == emp_id]
        model.Add(sum(emp_vars) <= int(emp["max_shifts"]))

        total_var = model.NewIntVar(0, max_possible_assignments, f"total_{emp_id}")
        model.Add(total_var == sum(emp_vars))
        total_assigned_per_emp[emp_id] = total_var

    # Forced assignments
    for assignment in forced_assignments:
        if assignment not in x:
            raise ValueError(f"Forced assignment does not exist in model: {assignment}")
        model.Add(x[assignment] == 1)

    # Fairness
    max_load = model.NewIntVar(0, max_possible_assignments, "max_load")
    for total_var in total_assigned_per_emp.values():
        model.Add(total_var <= max_load)

    # Preserve old assignments
    keep_vars: List[cp_model.IntVar] = []
    if preferred_schedule:
        for key, var in x.items():
            if key in preferred_assignments:
                keep_vars.append(var)

    penalty_terms: List[cp_model.LinearExpr] = []

    # Preferences
    for i, pref in enumerate(preferences):
        pref_type = pref["type"]
        employee_name = pref["employee_name"].lower()

        if employee_name not in name_to_id:
            continue

        emp_id = name_to_id[employee_name]
        penalty = int(pref.get("penalty", 1))

        if pref_type == "avoid_shift":
            target_shift = pref.get("shift")
            if not target_shift:
                continue

            for role in all_roles:
                for day in all_days:
                    key = (emp_id, day, target_shift, role)
                    if key in x:
                        penalty_terms.append(x[key] * penalty)

        elif pref_type == "avoid_back_to_back":
            for day in all_days:
                morning_vars = [
                    x[(emp_id, day, "morning", role)]
                    for role in all_roles
                    if (emp_id, day, "morning", role) in x
                ]
                evening_vars = [
                    x[(emp_id, day, "evening", role)]
                    for role in all_roles
                    if (emp_id, day, "evening", role) in x
                ]

                if not morning_vars or not evening_vars:
                    continue

                morning_assigned = model.NewBoolVar(f"morning_assigned_{emp_id}_{day}_{i}")
                evening_assigned = model.NewBoolVar(f"evening_assigned_{emp_id}_{day}_{i}")
                back_to_back = model.NewBoolVar(f"back_to_back_{emp_id}_{day}_{i}")

                model.Add(sum(morning_vars) >= 1).OnlyEnforceIf(morning_assigned)
                model.Add(sum(morning_vars) == 0).OnlyEnforceIf(morning_assigned.Not())

                model.Add(sum(evening_vars) >= 1).OnlyEnforceIf(evening_assigned)
                model.Add(sum(evening_vars) == 0).OnlyEnforceIf(evening_assigned.Not())

                model.AddBoolAnd([morning_assigned, evening_assigned]).OnlyEnforceIf(back_to_back)
                model.AddBoolOr([morning_assigned.Not(), evening_assigned.Not()]).OnlyEnforceIf(back_to_back.Not())

                penalty_terms.append(back_to_back * penalty)

    # Busy shift penalty for part-time staff
    busy_shift_penalty_terms: List[cp_model.LinearExpr] = []
    busy_penalty_weight = 4

    for (emp_id, day, shift, role), var in x.items():
        emp_type = staff_lookup[emp_id]["employment_type"]
        if is_busy_shift(day, shift) and emp_type != "full-time":
            busy_shift_penalty_terms.append(var * busy_penalty_weight)

    total_penalty = model.NewIntVar(0, 50000, "total_penalty")
    all_penalties = penalty_terms + busy_shift_penalty_terms

    if all_penalties:
        model.Add(total_penalty == sum(all_penalties))
    else:
        model.Add(total_penalty == 0)

    # Objective priority
    if keep_vars:
        model.Maximize(sum(keep_vars) * 1000 - total_penalty * 10 - max_load)
    else:
        model.Minimize(total_penalty * 10 + max_load)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise ValueError("No feasible schedule found under current constraints.")

    schedule: List[Dict[str, str]] = []
    for (emp_id, day, shift, role), var in x.items():
        if solver.Value(var) == 1:
            schedule.append(
                {
                    "employee_id": emp_id,
                    "employee_name": staff_lookup[emp_id]["name"],
                    "day": day,
                    "shift": shift,
                    "role": role,
                }
            )

    schedule.sort(key=lambda r: (r["day"], r["shift"], r["role"], r["employee_name"]))
    return schedule


def generate_schedule(data: Dict[str, Any]) -> List[Dict[str, str]]:
    return solve_schedule(data, preferred_schedule=None)


def print_schedule(schedule: List[Dict[str, str]], title: str = "SCHEDULE") -> None:
    day_order = {
        "Monday": 1,
        "Tuesday": 2,
        "Wednesday": 3,
        "Thursday": 4,
        "Friday": 5,
        "Saturday": 6,
        "Sunday": 7,
    }

    shift_order = {
        "morning": 1,
        "evening": 2,
    }

    grouped: Dict[Tuple[str, str], List[Dict[str, str]]] = defaultdict(list)
    for row in schedule:
        grouped[(row["day"], row["shift"])].append(row)

    print(f"\n===== {title} =====")
    for day, shift in sorted(
        grouped.keys(),
        key=lambda x: (day_order.get(x[0], 999), shift_order.get(x[1], 999))
    ):
        print(f"\n{day} - {shift}")
        for row in sorted(grouped[(day, shift)], key=lambda r: (r["role"], r["employee_name"])):
            print(f"  {row['role']:<8} -> {row['employee_name']} ({row['employee_id']})")
    print("=" * (len(title) + 12))
    print()


def print_preferences(data: Dict[str, Any]) -> None:
    prefs = data.get("preferences", [])
    print("\n===== CURRENT PREFERENCES =====")
    if not prefs:
        print("No preferences.")
    else:
        for i, pref in enumerate(prefs, start=1):
            print(f"{i}. {pref}")
    print("===============================\n")


def compare_schedules(old_schedule: List[Dict[str, str]], new_schedule: List[Dict[str, str]]) -> None:
    old_set = schedule_to_assignment_set(old_schedule)
    new_set = schedule_to_assignment_set(new_schedule)

    removed = old_set - new_set
    added = new_set - old_set

    print("===== CHANGES =====")
    if not removed and not added:
        print("No changes.")
    else:
        if removed:
            print("Removed assignments:")
            for emp_id, day, shift, role in sorted(removed):
                print(f"  - {emp_id}: {day} {shift} {role}")
        if added:
            print("Added assignments:")
            for emp_id, day, shift, role in sorted(added):
                print(f"  + {emp_id}: {day} {shift} {role}")
    print("===================\n")


def get_removed_and_added_assignments(
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    old_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in old_schedule
    }
    new_set = {
        (row["employee_id"], row["employee_name"], row["day"], row["shift"], row["role"])
        for row in new_schedule
    }

    removed = old_set - new_set
    added = new_set - old_set

    removed_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(removed)
    ]

    added_rows = [
        {
            "employee_id": emp_id,
            "employee_name": emp_name,
            "day": day,
            "shift": shift,
            "role": role,
        }
        for emp_id, emp_name, day, shift, role in sorted(added)
    ]

    return removed_rows, added_rows


def count_busy_assignments_for_employee(schedule: List[Dict[str, str]], employee_name: str) -> int:
    count = 0
    for row in schedule:
        if row["employee_name"].lower() == employee_name.lower() and is_busy_shift(row["day"], row["shift"]):
            count += 1
    return count


def generate_explanation(
    data_before: Dict[str, Any],
    old_schedule: List[Dict[str, str]],
    new_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> str:
    change_type = change.get("type", "").strip()
    removed_rows, added_rows = get_removed_and_added_assignments(old_schedule, new_schedule)
    lines: List[str] = []

    if change_type == "unavailable":
        employee_name = change["employee_name"]
        day = change["day"]
        shift = change["shift"]

        lines.append(
            f"{employee_name} was removed because they are unavailable for {day} {shift}."
        )

        replacement_rows = [
            row for row in added_rows
            if row["day"].lower() == day.lower() and row["shift"].lower() == shift.lower()
        ]

        if replacement_rows:
            for row in replacement_rows:
                emp_info = next(
                    (s for s in data_before["staff"] if s["name"].lower() == row["employee_name"].lower()),
                    None
                )
                if emp_info and emp_info["employment_type"] == "full-time" and is_busy_shift(row["day"], row["shift"]):
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available, qualified as a {row['role']}, and full-time staff are preferred on busy shifts."
                    )
                else:
                    lines.append(
                        f"{row['employee_name']} was assigned because they are available and qualified as a {row['role']}."
                    )

        if not replacement_rows:
            lines.append(
                "The schedule was re-optimized to keep the shift covered with available qualified staff."
            )

    elif change_type == "direct_swap":
        employee_name_1 = change["employee_name_1"]
        day_1 = change["day_1"]
        shift_1 = change["shift_1"]
        employee_name_2 = change["employee_name_2"]
        day_2 = change["day_2"]
        shift_2 = change["shift_2"]

        lines.append(
            f"A direct swap was performed between {employee_name_1} ({day_1} {shift_1}) and {employee_name_2} ({day_2} {shift_2})."
        )
        lines.append(
            "The swap was accepted because both employees were assigned to those shifts and remained qualified for the exchanged roles."
        )

    elif change_type == "avoid_back_to_back":
        employee_name = change["employee_name"]
        before_count = 0
        after_count = 0

        for day in {row["day"] for row in old_schedule + new_schedule}:
            old_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "morning"
                for r in old_schedule
            )
            old_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in old_schedule
            )
            new_morning = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "morning"
                for r in new_schedule
            )
            new_evening = any(
                r["employee_name"].lower() == employee_name.lower() and r["day"] == day and r["shift"] == "evening"
                for r in new_schedule
            )

            if old_morning and old_evening:
                before_count += 1
            if new_morning and new_evening:
                after_count += 1

        lines.append(
            f"A preference was added to avoid assigning {employee_name} to both morning and evening on the same day."
        )
        lines.append(
            f"Back-to-back assignments for {employee_name} changed from {before_count} to {after_count}."
        )
        lines.append(
            "The schedule was re-optimized to reduce back-to-back assignments when possible."
        )

    elif change_type == "avoid_shift":
        employee_name = change["employee_name"]
        shift = change["shift"]

        before_count = sum(
            1 for row in old_schedule
            if row["employee_name"].lower() == employee_name.lower() and row["shift"].lower() == shift.lower()
        )
        after_count = sum(
            1 for row in new_schedule
            if row["employee_name"].lower() == employee_name.lower() and row["shift"].lower() == shift.lower()
        )

        lines.append(
            f"A preference was added to avoid assigning {employee_name} to {shift} shifts."
        )
        lines.append(
            f"{shift.capitalize()} assignments for {employee_name} changed from {before_count} to {after_count}."
        )
        lines.append(
            f"The schedule was re-optimized to reduce {shift} assignments for {employee_name} when possible."
        )

    else:
        lines.append("The schedule was updated and re-optimized based on the requested change.")

    # Busy shift explanation
    full_time_staff = {
        s["name"] for s in data_before["staff"] if s["employment_type"] == "full-time"
    }

    busy_added = [row for row in added_rows if is_busy_shift(row["day"], row["shift"])]
    if busy_added:
        full_time_busy_added = [row for row in busy_added if row["employee_name"] in full_time_staff]
        if full_time_busy_added:
            names = ", ".join(sorted({row["employee_name"] for row in full_time_busy_added}))
            lines.append(
                f"Busy shifts prefer full-time staff when possible, which influenced assignments such as {names}."
            )
        else:
            lines.append(
                "Busy shifts prefer full-time staff when possible, but availability and skill constraints still had to be satisfied."
            )

    if removed_rows or added_rows:
        lines.append("Other assignments were kept unchanged when possible.")

    return "\n".join(lines)


def print_explanation(explanation: str) -> None:
    print("===== EXPLANATION =====")
    print(explanation)
    print("=======================\n")


def update_schedule(
    data: Dict[str, Any],
    existing_schedule: List[Dict[str, str]],
    change: Dict[str, Any],
) -> Tuple[List[Dict[str, str]], Dict[str, Any]]:
    change_type = change.get("type", "").strip()

    updated_data = {
        "staff": [dict(s) for s in data["staff"]],
        "availability": [dict(a) for a in data["availability"]],
        "shift_requirements": [dict(r) for r in data["shift_requirements"]],
        "preferences": [dict(p) for p in data.get("preferences", [])],
    }

    if change_type == "unavailable":
        employee_name = change["employee_name"].strip()
        day = change["day"].strip()
        shift = change["shift"].strip()

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        emp_id = matched[0]["id"]
        set_availability(updated_data, emp_id, day, shift, False)

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "direct_swap":
        name_1 = change["employee_name_1"].strip()
        day_1 = change["day_1"].strip()
        shift_1 = change["shift_1"].strip()

        name_2 = change["employee_name_2"].strip()
        day_2 = change["day_2"].strip()
        shift_2 = change["shift_2"].strip()

        matched_1 = [s for s in data["staff"] if s["name"].lower() == name_1.lower()]
        matched_2 = [s for s in data["staff"] if s["name"].lower() == name_2.lower()]

        if not matched_1:
            raise ValueError(f"Employee '{name_1}' not found.")
        if not matched_2:
            raise ValueError(f"Employee '{name_2}' not found.")

        emp_1 = matched_1[0]
        emp_2 = matched_2[0]

        if emp_1["id"] == emp_2["id"]:
            raise ValueError("direct_swap requires two different employees.")

        roles_1 = get_employee_assignment_roles(existing_schedule, name_1, day_1, shift_1)
        roles_2 = get_employee_assignment_roles(existing_schedule, name_2, day_2, shift_2)

        if not roles_1:
            raise ValueError(f"{name_1} is not currently assigned to {day_1} {shift_1}.")
        if not roles_2:
            raise ValueError(f"{name_2} is not currently assigned to {day_2} {shift_2}.")
        if len(roles_1) != 1 or len(roles_2) != 1:
            raise ValueError("Each employee must have exactly one role in their specified shift.")

        role_1 = roles_1[0]
        role_2 = roles_2[0]

        if role_2 not in set(emp_1["skills"]):
            raise ValueError(
                f"{name_1} cannot take {name_2}'s shift because {name_1} lacks skill '{role_2}'."
            )
        if role_1 not in set(emp_2["skills"]):
            raise ValueError(
                f"{name_2} cannot take {name_1}'s shift because {name_2} lacks skill '{role_1}'."
            )

        set_availability(updated_data, emp_1["id"], day_2, shift_2, True)
        set_availability(updated_data, emp_2["id"], day_1, shift_1, True)

        forced_assignments = [
            (emp_1["id"], day_2, shift_2, role_2),
            (emp_2["id"], day_1, shift_1, role_1),
        ]

        updated_schedule = solve_schedule(
            updated_data,
            preferred_schedule=existing_schedule,
            forced_assignments=forced_assignments,
        )
        return updated_schedule, updated_data

    if change_type == "avoid_back_to_back":
        employee_name = change["employee_name"].strip()
        penalty = int(change.get("penalty", 5))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_back_to_back",
                "employee_name": employee_name,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    if change_type == "avoid_shift":
        employee_name = change["employee_name"].strip()
        target_shift = change["shift"].strip()
        penalty = int(change.get("penalty", 3))

        matched = [s for s in data["staff"] if s["name"].lower() == employee_name.lower()]
        if not matched:
            raise ValueError(f"Employee '{employee_name}' not found.")

        add_preference(
            updated_data,
            {
                "type": "avoid_shift",
                "employee_name": employee_name,
                "shift": target_shift,
                "penalty": penalty,
            },
        )

        updated_schedule = solve_schedule(updated_data, preferred_schedule=existing_schedule)
        return updated_schedule, updated_data

    raise ValueError(
        "Currently supported update types are 'unavailable', 'direct_swap', 'avoid_back_to_back', and 'avoid_shift'."
    )


def prompt_change_request() -> Dict[str, Any]:
    print("\nChoose an update type:")
    print("1. unavailable")
    print("2. direct_swap")
    print("3. avoid_back_to_back")
    print("4. avoid_shift")

    while True:
        choice = input("Enter 1, 2, 3, or 4: ").strip()
        if choice in {"1", "2", "3", "4"}:
            break
        print("Please enter 1, 2, 3, or 4.")

    if choice == "1":
        employee_name = input("Employee name: ").strip()
        day = input("Day (e.g. Monday/Saturday): ").strip()
        shift = input("Shift (e.g. morning/evening): ").strip()

        return {
            "type": "unavailable",
            "employee_name": employee_name,
            "day": day,
            "shift": shift,
        }

    if choice == "2":
        employee_name_1 = input("Employee 1 name: ").strip()
        day_1 = input("Employee 1 current day: ").strip()
        shift_1 = input("Employee 1 current shift: ").strip()

        employee_name_2 = input("Employee 2 name: ").strip()
        day_2 = input("Employee 2 current day: ").strip()
        shift_2 = input("Employee 2 current shift: ").strip()

        return {
            "type": "direct_swap",
            "employee_name_1": employee_name_1,
            "day_1": day_1,
            "shift_1": shift_1,
            "employee_name_2": employee_name_2,
            "day_2": day_2,
            "shift_2": shift_2,
        }

    if choice == "3":
        employee_name = input("Employee name: ").strip()
        penalty_text = input("Penalty weight (default 5): ").strip()
        penalty = int(penalty_text) if penalty_text else 5

        return {
            "type": "avoid_back_to_back",
            "employee_name": employee_name,
            "penalty": penalty,
        }

    employee_name = input("Employee name: ").strip()
    shift = input("Shift to avoid (e.g. evening): ").strip()
    penalty_text = input("Penalty weight (default 3): ").strip()
    penalty = int(penalty_text) if penalty_text else 3

    return {
        "type": "avoid_shift",
        "employee_name": employee_name,
        "shift": shift,
        "penalty": penalty,
    }


def ask_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()
        if answer in {"y", "yes"}:
            return True
        if answer in {"n", "no"}:
            return False
        print("Please enter y or n.")


def main() -> None:
    data_file = "/content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json"

    try:
        data = load_data_from_json(data_file)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error loading data: {e}")
        return

    print(f"Loaded data from {data_file}")
    print("Generating initial schedule...")

    try:
        schedule = generate_schedule(data)
    except ValueError as e:
        print(f"Scheduling error: {e}")
        return

    print_schedule(schedule, title="INITIAL SCHEDULE")
    print_preferences(data)

    while True:
        should_update = ask_yes_no("Do you want to enter an update? (y/n): ")
        if not should_update:
            print("Done.")
            break

        try:
            change_request = prompt_change_request()
            updated_schedule, updated_data = update_schedule(data, schedule, change_request)

            print_schedule(updated_schedule, title="UPDATED SCHEDULE")
            compare_schedules(schedule, updated_schedule)

            explanation = generate_explanation(data, schedule, updated_schedule, change_request)
            print_explanation(explanation)

            print_preferences(updated_data)

            data = updated_data
            schedule = updated_schedule

            save_changes = ask_yes_no("Do you want to save updates back to JSON? (y/n): ")
            if save_changes:
                save_data_to_json(data, data_file)
                print(f"Saved updated data to {data_file}")

        except ValueError as e:
            print(f"Error: {e}")

        show_current = ask_yes_no("Do you want to print the current final schedule again? (y/n): ")
        if show_current:
            print_schedule(schedule, title="CURRENT FINAL SCHEDULE")
            print_preferences(data)


if __name__ == "__main__":
    main()

Loaded data from /content/drive/MyDrive/Colab Notebooks/stat541/restaurant_data.json
Generating initial schedule...

===== INITIAL SCHEDULE =====

Monday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Monday - evening
  cashier  -> Brian (E02)
  server   -> Eric (E05)
  server   -> George (E07)

Tuesday - morning
  cashier  -> Julia (E10)
  server   -> Daniel (E04)

Tuesday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> George (E07)

Wednesday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Wednesday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Thursday - morning
  cashier  -> Fiona (E06)
  server   -> Hannah (E08)

Thursday - evening
  cashier  -> Ian (E09)
  server   -> Brian (E02)
  server   -> Eric (E05)

Friday - morning
  cashier  -> Hannah (E08)
  server   -> Alice (E01)

Friday - evening
  cashier  -> Ian (E09)
  server   -> Eric (E05)
  server   -> George (E07)

Saturday - mor